In [16]:
!pip install transformers torch seqeval -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
data = [
    (["John", "works", "at", "Google"], ["NOUN", "VERB", "ADP", "PROPN"]),
    (["She", "lives", "in", "India"], ["PRON", "VERB", "ADP", "PROPN"]),
    (["I", "love", "machine", "learning"], ["PRON", "VERB", "NOUN", "NOUN"]),
]

In [18]:
label_list = list(set(label for _, labels in data for label in labels))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [19]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [21]:
def tokenize_and_align(data):
    encodings = []
    
    for tokens, labels in data:
        tokenized = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")
        word_ids = tokenized.word_ids()

        aligned_labels = []
        prev = None

        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != prev:
                aligned_labels.append(label2id[labels[word_idx]])
            else:
                aligned_labels.append(-100)
            prev = word_idx

        encodings.append((tokenized, aligned_labels))
    
    return encodings

encoded_data = tokenize_and_align(data)

In [22]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
import torch
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

model.train()

for epoch in range(3):
    for inputs, labels in encoded_data:
        outputs = model(**inputs, labels=torch.tensor([labels]))
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    print(f"Epoch {epoch} Loss:", loss.item())

Epoch 0 Loss: 1.6098389625549316
Epoch 1 Loss: 1.2261755466461182
Epoch 2 Loss: 0.920794665813446


In [24]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs)
    preds = outputs.logits.argmax(dim=2)

    return list(zip(tokens, [id2label[p.item()] for p in preds[0]]))

print(predict("John works at Google"))

[('John', 'PROPN'), ('works', 'NOUN'), ('at', 'VERB'), ('Google', 'ADP')]
